# Classifier-Training (Colab)

Trainiert den Package-Classifier (`medrax.training`, ResNet18 + eigener Head) → Checkpoint nach
`weights/classifier/best_model.pt` (Drive), den der Agent lädt.

Decoding und Training sind getrennt: DICOM→PNG einmalig als Zip auf Drive, Training liest lokal von `/content`.

> Runtime = GPU (T4). Forschungs-/Entwicklungsprojekt, nicht klinisch validiert.

## 0) Setup (Drive mounten, ins Projekt wechseln, Minimal-Deps)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# >>> Bei Bedarf den Projektpfad anpassen (siehe notebooks/colab_notes.md) <<<
%cd /content/drive/MyDrive/Colab/IKIM_CXR

# Minimal-Abhängigkeiten fürs Training (Core ist in Colab meist vorhanden).
# pydicom + pylibjpeg werden für das DICOM-Decoding (JPEG2000) benötigt:
!pip -q install pydicom pylibjpeg pylibjpeg-openjpeg pylibjpeg-libjpeg python-dotenv scikit-image

## 1) Konfiguration & Hyperparameter

Nur diese Zelle musst du ggf. anpassen.

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
import config
config.ensure_dirs()
print(config.summary())
print('Device:', config.DEVICE, '(sollte cuda sein)')

# === Hyperparameter (anpassen) ===
CACHE_SIZE = 512          # Auflösung der gecachten PNGs (einmaliger Decode-Schritt)
IMG_SIZE   = 384          # Trainingsauflösung
BATCH      = 32           # bei mehr VRAM erhöhen (z. B. 64)
EPOCHS     = 10
BACKBONE   = 'resnet18'   # oder 'efficientnet_b0'
WORKERS    = 4

TRAIN_DIR = str(config.TRAIN_DIR)                                    # Drive: data/VinBigData/train
TRAIN_CSV = str(config.TRAIN_CSV)                                    # Drive: data/VinBigData/train.csv
DRIVE_ZIP = str(config.PROCESSED_DIR / f'train_png_{CACHE_SIZE}.zip')
LOCAL_PNG = '/content/train_png'
print('TRAIN_DIR:', TRAIN_DIR)
print('TRAIN_CSV:', TRAIN_CSV)
print('DRIVE_ZIP:', DRIVE_ZIP)

## 2) DICOM → PNG (einmalig, Zip auf Drive)

Läuft nur, wenn das Zip fehlt. Decode der 15.000 DICOMs dauert einmalig, danach nicht mehr.

In [ ]:
import os, shutil
if os.path.exists(DRIVE_ZIP):
    print('Cache-Zip existiert bereits – Decoding übersprungen:', DRIVE_ZIP)
else:
    assert os.path.isdir(TRAIN_DIR), f'Train-Ordner fehlt: {TRAIN_DIR} (DICOMs nach Drive hochladen)'
    # 1) DICOM -> verkleinerte PNG (parallel) nach lokalem /content
    !python scripts/convert_dicom_to_png_cache.py --src "{TRAIN_DIR}" --dst "{LOCAL_PNG}" --size {CACHE_SIZE} --workers {WORKERS}
    # 2) Als EIN Zip auf Drive persistieren (zip -0 = store, schnell, da PNG bereits komprimiert)
    os.makedirs(os.path.dirname(DRIVE_ZIP), exist_ok=True)
    !cd /content && zip -0 -r -q train_png.zip train_png
    shutil.move('/content/train_png.zip', DRIVE_ZIP)
    print('Cache-Zip auf Drive gespeichert:', DRIVE_ZIP)

## 3) PNG-Cache lokal bereitstellen (pro Session, schnell)

Kopiert das Zip von Drive nach `/content` und entpackt es – Training liest dann vom schnellen lokalen Disk.

In [ ]:
import os
if not os.path.isdir(LOCAL_PNG) or len(os.listdir(LOCAL_PNG)) == 0:
    print('Hole PNG-Cache von Drive ...')
    !cp "{DRIVE_ZIP}" /content/train_png.zip
    !cd /content && unzip -q -o train_png.zip
print('Anzahl PNGs lokal:', len(os.listdir(LOCAL_PNG)))

## 4) Training (Package-Classifier, GPU + Mixed Precision)

Liest die Labels aus `train.csv` (Detection → image-level Multi-Label, automatisch) und die Bilder aus dem lokalen PNG-Cache.

In [ ]:
!python -m medrax.training.train_classifier --image-dir "{LOCAL_PNG}" --csv "{TRAIN_CSV}" --backbone {BACKBONE} --img-size {IMG_SIZE} --batch-size {BATCH} --num-workers {WORKERS} --amp --epochs {EPOCHS}

## 5) Verify + Übergabe an den Agenten

Prüft den gespeicherten Checkpoint und macht eine Beispiel-Vorhersage über dasselbe Tool, das der Agent nutzt.

In [ ]:
import glob, os
from medrax.utils.paths import find_classifier_checkpoint
ckpt = find_classifier_checkpoint()
print('Gespeicherter Checkpoint:', ckpt)

from medrax.tools.my_classifier_tool import MyChestXRayClassifierTool
tool = MyChestXRayClassifierTool(img_size=IMG_SIZE)
sample = sorted(glob.glob(LOCAL_PNG + '/*.png'))[0]
res = tool.analyze(sample)
print('Beispielbild:', os.path.basename(sample))
print('Top-5:', res['top'])
print('Diesen Checkpoint nutzt der Agent:', res['metadata']['model_checkpoint'])
print('')
print('Fertig – der Agent kann jetzt mit diesem Classifier laufen.')